# Generative Adversarial Networks

In this notebook, we study Generative Adversarial Networks, or GANs.

GANs are generative models. Their goal is to learn a data distribution and generate new samples that look similar to the training data.

Instead of directly estimating the probability of each data point, GANs train two neural networks against each other:

1. **Generator**
   Takes a random latent vector \(z\) and creates a fake sample.

2. **Discriminator**
   Receives real and fake samples and learns to distinguish them.

The generator tries to fool the discriminator.
The discriminator tries to correctly classify real and fake samples.

This adversarial setup can produce realistic samples, but GAN training can also be unstable.

In this notebook, we will implement a simple GAN for image generation and investigate:

1. how the generator and discriminator interact
2. how GAN losses are computed
3. how alternating training works
4. why visual inspection is important
5. how the latent space can be explored
6. what can go wrong during GAN training

## 0. Generative Modeling Motivation

So far, many models we trained were **discriminative models**.

A discriminative model learns to predict a label from an input.

Example:

$$
x \rightarrow y
$$

For instance:

```text
image → digit class
```
A generative model has a different goal.
It learns to generate new samples that look like they could come from the training data.

The goal can be written as:

$$
p_{\text{model}}(x) \approx p_{\text{data}}(x)
$$

This means that the model distribution should become similar to the real data distribution.

For example, after training on handwritten digits, a generative model should create new digit-like images that were not copied directly from the training set.

## 1. The GAN Game: Generator vs. Discriminator

A GAN consists of two neural networks that are trained together.

### Generator

The generator receives a random latent vector \(z\) and creates a fake sample:

$$
\text{fake image} = G(z)
$$

At the beginning of training, the generated samples look like noise.
During training, the generator improves because it receives feedback from the discriminator.

### Discriminator

The discriminator receives an image and predicts whether it is real or fake:

$$
D(x) \rightarrow \text{real/fake score}
$$

It sees two kinds of inputs:

1. real images from the training dataset
2. fake images created by the generator

The discriminator learns to classify real images as real and generated images as fake.

### Adversarial Training

The two networks have opposite goals:

- the discriminator tries to correctly distinguish real from fake samples
- the generator tries to fool the discriminator into classifying fake samples as real

This creates an adversarial training process.

### Task 1: Create Latent Noise Vectors

The generator does not receive an image as input.
It receives a random latent vector \(z\).

Usually, \(z\) is sampled from a simple distribution such as a standard normal distribution:

$$
z \sim \mathcal{N}(0, 1)
$$

Complete the function below.

In [ ]:
import torch

def sample_noise(batch_size, latent_dim, device):
    """
    Sample random latent vectors for the generator.

    Args:
        batch_size: number of latent vectors
        latent_dim: size of each latent vector
        device: torch device

    Returns:
        Tensor of shape (batch_size, latent_dim)
    """

    # TODO: sample from a standard normal distribution
    z = ...

    return z

In [ ]:
# Test your code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 8
latent_dim = 100

z = sample_noise(batch_size, latent_dim, device)

print("Latent noise shape:", z.shape)
print("Mean:", z.mean().item())
print("Standard deviation:", z.std().item())

# Expected idea:
# Latent noise shape: torch.Size([8, 100])
# Mean: close to 0
# Standard deviation: close to 1

**Questions**

1. Why does the generator start from random noise?
2. What does `latent_dim` control?
3. Why do we sample a batch of latent vectors instead of only one vector?

## 2. GAN Losses

GANs train two networks with different objectives.

The discriminator receives real and fake images and tries to classify them correctly.

For real images, the target label is:

$$
1
$$

For fake images, the target label is:

$$
0
$$

The generator wants the discriminator to classify fake images as real.
Therefore, when training the generator, fake images receive the target label:

$$
1
$$

In this notebook, we use `BCEWithLogitsLoss`.

This means the discriminator should output **raw logits**, not sigmoid probabilities.

In [ ]:
import torch
import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

### Task 2: Compute Discriminator Loss

In [ ]:
def discriminator_loss(real_logits, fake_logits):
    """
    Compute discriminator loss.

    Args:
        real_logits: discriminator outputs for real images
        fake_logits: discriminator outputs for fake images

    Returns:
        discriminator loss
    """

    # TODO: create labels for real images
    real_labels = ...

    # TODO: create labels for fake images
    fake_labels = ...

    # TODO: compute loss for real images
    real_loss = ...

    # TODO: compute loss for fake images
    fake_loss = ...

    # TODO: combine both losses
    loss = ...

    return loss

### Task 3: Compute the Generator Loss

In [ ]:
def generator_loss(fake_logits):
    """
    Compute generator loss.

    Args:
        fake_logits: discriminator outputs for generated images

    Returns:
        generator loss
    """

    # TODO: create labels that represent "real"
    real_labels = ...

    # TODO: compute generator loss
    loss = ...

    return loss

In [ ]:
real_logits = torch.tensor([[2.0], [1.5], [3.0]])
fake_logits = torch.tensor([[-2.0], [-1.0], [-3.0]])

d_loss = discriminator_loss(real_logits, fake_logits)
g_loss = generator_loss(fake_logits)

print("Discriminator loss:", d_loss.item())
print("Generator loss:", g_loss.item())

In this example, the discriminator is doing well:

- real images get positive logits
- fake images get negative logits

Therefore, the discriminator loss should be low.

The generator loss is high because the discriminator correctly identifies the fake samples as fake.

## 3. Generator Network

The generator transforms a random latent vector into an image.

The input is random noise:

$$
z \sim \mathcal{N}(0, 1)
$$

The output should have the same shape as the training images.

For MNIST or FashionMNIST, each image has shape:

$$
1 \times 28 \times 28
$$

In this notebook, we start with a simple fully connected generator.

The generator maps:

$$
z \rightarrow \text{fake image}
$$

Because the images will be normalized to the range `[-1, 1]`, the generator uses `Tanh` as the final activation.

In [ ]:
# Load dataset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

In [ ]:
def show_images(images, title):
    images = images.detach().cpu()

    # Convert from [-1, 1] back to [0, 1]
    images = (images + 1) / 2

    fig, axes = plt.subplots(1, 8, figsize=(12, 2))

    for i, ax in enumerate(axes):
        ax.imshow(images[i].squeeze(), cmap="gray")
        ax.axis("off")

    plt.suptitle(title)
    plt.show()


real_images, real_labels = next(iter(train_loader))

show_images(real_images[:8], "Real MNIST Images")

### Task 4: Implement the Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim, image_dim):
        super().__init__()

        self.model = nn.Sequential(
            # TODO: map latent vector to hidden representation
            ...,

            # TODO: add activation
            ...,

            # TODO: add another hidden layer
            ...,

            # TODO: add activation
            ...,

            # TODO: map to image dimension
            ...,

            # TODO: constrain output to [-1, 1]
            ...
        )

    def forward(self, z):
        image = self.model(z)

        # TODO: reshape flat vector into image
        image = ...

        return image

In [ ]:
latent_dim = 100
image_dim = 28 * 28

generator = Generator(latent_dim, image_dim).to(device)

z = sample_noise(batch_size=8, latent_dim=latent_dim, device=device)

fake_images = generator(z)

print("Noise shape:", z.shape)
print("Generated image shape:", fake_images.shape)

show_images(fake_images, "Generated Images Before Training")

**Question:**
Why do we reshape the output into `(batch_size, 1, 28, 28)`?

## 4. Discriminator Network

The discriminator receives an image and predicts whether it is real or fake.

It maps:

$$
x \rightarrow \text{real/fake logit}
$$

For each image, the discriminator outputs one value.

A large positive logit means the discriminator thinks the image is real.
A large negative logit means the discriminator thinks the image is fake.

Because we use `BCEWithLogitsLoss`, the discriminator should output raw logits, not sigmoid probabilities.

### Task 5: Implement the Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, image_dim):
        super().__init__()

        self.model = nn.Sequential(
            # TODO: map flattened image to hidden representation
            ...,

            # TODO: add activation
            ...,

            # TODO: add another hidden layer
            ...,

            # TODO: add activation
            ...,

            # TODO: output one real/fake logit
            ...
        )

    def forward(self, image):
        # TODO: flatten image
        image = ...

        # TODO: compute real/fake logit
        logit = ...

        return logit

In [ ]:
discriminator = Discriminator(image_dim).to(device)

real_images, _ = next(iter(train_loader))
real_images = real_images.to(device)

z = sample_noise(batch_size=real_images.size(0), latent_dim=latent_dim, device=device)
fake_images = generator(z)

real_logits = discriminator(real_images)
fake_logits = discriminator(fake_images)

print("Real image batch shape:", real_images.shape)
print("Fake image batch shape:", fake_images.shape)
print("Real logits shape:", real_logits.shape)
print("Fake logits shape:", fake_logits.shape)

In [ ]:
real_probabilities = torch.sigmoid(real_logits)
fake_probabilities = torch.sigmoid(fake_logits)

print("Mean probability for real images:", real_probabilities.mean().item())
print("Mean probability for fake images:", fake_probabilities.mean().item())

At this point, the generator and discriminator are not trained yet.

The discriminator's probabilities may be close to random because its weights are still randomly initialized.

## 5. Alternating GAN Training

GANs are trained by alternating between two updates:

1. **Update the discriminator**
   - use real images from the dataset
   - use fake images from the generator
   - train the discriminator to classify real as real and fake as fake

2. **Update the generator**
   - generate fake images
   - pass them through the discriminator
   - train the generator so that fake images are classified as real

When training the discriminator, fake images should be detached from the generator graph.

When training the generator, fake images must not be detached, because gradients need to flow back into the generator.

### Task 6: Implement One GAN Training step

In [ ]:
def train_gan_step(
    generator,
    discriminator,
    real_images,
    latent_dim,
    optimizer_g,
    optimizer_d,
    device
):
    batch_size = real_images.size(0)

    real_images = real_images.to(device)

    # -------------------------
    # 1. Train Discriminator
    # -------------------------

    optimizer_d.zero_grad()

    # TODO: compute discriminator logits for real images
    real_logits = ...

    # TODO: sample latent noise
    z = ...

    # TODO: generate fake images
    fake_images = ...

    # TODO: compute discriminator logits for fake images
    # Important: detach fake images here
    fake_logits = ...

    # TODO: compute discriminator loss
    d_loss = ...

    # TODO: update discriminator
    ...

    # -------------------------
    # 2. Train Generator
    # -------------------------

    optimizer_g.zero_grad()

    # TODO: sample new latent noise
    z = ...

    # TODO: generate fake images
    fake_images = ...

    # TODO: compute discriminator logits for fake images
    # Important: do NOT detach fake images here
    fake_logits = ...

    # TODO: compute generator loss
    g_loss = ...

    # TODO: update generator
    ...

    return d_loss.item(), g_loss.item()

In [ ]:
optimizer_g = torch.optim.Adam(
    generator.parameters(),
    lr=0.0002,
    betas=(0.5, 0.999)
)

optimizer_d = torch.optim.Adam(
    discriminator.parameters(),
    lr=0.0002,
    betas=(0.5, 0.999)
)

In [ ]:
num_epochs = 10

g_losses = []
d_losses = []

fixed_noise = sample_noise(batch_size=8, latent_dim=latent_dim, device=device)

for epoch in range(num_epochs):
    epoch_g_loss = 0
    epoch_d_loss = 0

    for real_images, _ in train_loader:
        d_loss, g_loss = train_gan_step(
            generator=generator,
            discriminator=discriminator,
            real_images=real_images,
            latent_dim=latent_dim,
            optimizer_g=optimizer_g,
            optimizer_d=optimizer_d,
            device=device
        )

        epoch_d_loss += d_loss
        epoch_g_loss += g_loss

    avg_d_loss = epoch_d_loss / len(train_loader)
    avg_g_loss = epoch_g_loss / len(train_loader)

    d_losses.append(avg_d_loss)
    g_losses.append(avg_g_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"D Loss: {avg_d_loss:.4f} | "
        f"G Loss: {avg_g_loss:.4f}"
    )

    with torch.no_grad():
        samples = generator(fixed_noise)
        show_images(samples, f"Generated Images After Epoch {epoch + 1}")

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(d_losses, label="Discriminator Loss")
plt.plot(g_losses, label="Generator Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.show()

The generated images should become more digit-like over time, but they will probably not look perfect.

This is expected: we use a very simple fully connected GAN. More advanced architectures such as DCGANs usually produce better image quality.

## 6. Latent Space Exploration

After training, the generator maps latent vectors to images:

$$
z \rightarrow G(z)
$$

If the GAN has learned a meaningful latent space, nearby latent vectors should often produce visually related images.

We can test this by interpolating between two latent vectors.

Instead of jumping directly from one random vector to another, we create intermediate vectors:

$$
z_{\alpha} = (1 - \alpha)z_1 + \alpha z_2
$$

where $\alpha$ gradually changes from 0 to 1.

This lets us observe how generated images change smoothly in latent space.

### Task 7: Interpolate Between Two Latent Vectors

In [ ]:
def interpolate_noise(z1, z2, num_steps):
    """
    Create interpolated latent vectors between z1 and z2.

    Args:
        z1: first latent vector, shape (1, latent_dim)
        z2: second latent vector, shape (1, latent_dim)
        num_steps: number of interpolation steps

    Returns:
        Tensor of shape (num_steps, latent_dim)
    """

    interpolated_vectors = []

    for alpha in torch.linspace(0, 1, num_steps):
        # TODO: interpolate between z1 and z2
        z = ...

        interpolated_vectors.append(z)

    interpolated_vectors = torch.cat(interpolated_vectors, dim=0)

    return interpolated_vectors

In [ ]:
generator.eval()

z1 = sample_noise(batch_size=1, latent_dim=latent_dim, device=device)
z2 = sample_noise(batch_size=1, latent_dim=latent_dim, device=device)

interpolated_z = interpolate_noise(
    z1=z1,
    z2=z2,
    num_steps=8
)

with torch.no_grad():
    interpolated_images = generator(interpolated_z)

show_images(interpolated_images, "Latent Space Interpolation")

**Questions**

1. Do the generated images change abruptly or smoothly?
2. What would smooth changes suggest about the latent space?
3. Why is interpolation useful for understanding generative models?

## 7. Failure Modes and Loss Interpretation

GAN losses are harder to interpret than ordinary classification losses because two networks are trained against each other.

A low discriminator loss usually means that the discriminator can easily distinguish real from fake samples. This often indicates that the generator is still weak.

A low generator loss also does not guarantee good samples. The generator may fool the discriminator with low-diversity outputs.

Therefore, generated images should always be inspected together with the loss curves.

One common GAN failure mode is **mode collapse**.

Mode collapse happens when the generator produces low-diversity samples, even though the real dataset contains many different modes.

For example, on MNIST, a collapsed generator might produce only images that look like the digit `1`, instead of generating all digit classes.

### Task 8: Measure Sample Diversity

In [ ]:
def image_diversity(images):
    """
    Compute a simple diversity score for a batch of images.

    Args:
        images: tensor of shape (batch_size, channels, height, width)

    Returns:
        scalar diversity score
    """

    # TODO: flatten images
    images = ...

    # TODO: compute standard deviation across the batch
    pixel_std = ...

    # TODO: average over all pixels
    diversity = ...

    return diversity

In [ ]:
generator.eval()

with torch.no_grad():
    z = sample_noise(batch_size=32, latent_dim=latent_dim, device=device)
    generated_images = generator(z)

# Create artificial mode collapse by repeating one image
collapsed_images = generated_images[0:1].repeat(32, 1, 1, 1)

generated_diversity = image_diversity(generated_images)
collapsed_diversity = image_diversity(collapsed_images)

print("Generated image diversity:", generated_diversity.item())
print("Collapsed image diversity:", collapsed_diversity.item())

In [ ]:
show_images(generated_images[:8], "Generated Images")
show_images(collapsed_images[:8], "Artificial Mode Collapse")

This diversity score is only a simple proxy. It can show whether all generated images are identical, but it is not a full evaluation metric for GAN quality.

## 8. Conditional and Advanced GAN Variants

The GAN we implemented is an unconditional GAN.

It generates samples from random noise:

$$
z \rightarrow G(z)
$$

However, many GAN variants add more structure or control to the generation process.

### Conditional GAN

A conditional GAN gives both the generator and discriminator additional information, such as a class label.

For example, instead of generating any digit, we can ask the generator to generate a specific digit:

$$
G(z, y)
$$

where \(y\) is the class label.

The discriminator also receives the label:

$$
D(x, y)
$$

This allows the model to learn class-specific generation.

### InfoGAN

InfoGAN tries to learn interpretable latent variables.

Instead of using only random noise, it adds structured latent codes:

$$
G(z, c)
$$

The goal is that parts of \(c\) control meaningful properties of the generated image, such as digit type, rotation, or thickness.

### StyleGAN

StyleGAN improves image generation by controlling style at different levels of the generator.

Coarse layers may affect high-level features such as pose or face shape.
Fine layers may affect details such as color, texture, or small visual patterns.

### CycleGAN

CycleGAN learns image-to-image translation between two domains without paired examples.

Examples:

- horse ↔ zebra
- summer ↔ winter
- photo ↔ painting style

It uses two generators and two discriminators to learn transformations in both directions.

### Wasserstein GAN

Wasserstein GAN changes the training objective.

Instead of using the standard GAN loss, it uses the Wasserstein distance, also called Earth Mover’s distance.

This can provide more stable gradients and more meaningful loss curves.

Feel free to check out other interesting notebooks with more complex GAN networks where you can play around with your own images: https://drive.google.com/drive/folders/1LBWcmnUPoHDeaYlRiHokGyjywIdyhAQb?usp=sharing